In [1]:
println("Loading imports")
include("../src/optim/gradient_descent.jl")
include("../src/optim/greedy_descent.jl")
include("../src/optim/loss.jl")

Loading imports


MPE_loss_init (generic function with 1 method)

In [2]:
println("Initialising data")

function pseudorandom_padic_int(p, prec, exp)
    K = PadicField(p, prec)
    coeffs = rand(0:(p-1), exp+1)
    return K(sum([coeffs[i] * p^(i-1) for i in Base.eachindex(coeffs)]))
end

# Initialise the p-adic field
prec = 20
p = 2
K = PadicField(p, prec)
poly_degree = 7

c = [pseudorandom_padic_int(p, prec, 5) for i in 1:poly_degree]
vars = ["x"]
append!(vars, ["a_"*string(i) for i in 1:poly_degree])
pts = [ValuationPolydisc{PadicFieldElem, Int}([c_i], [prec]) for c_i in c]
data = [(p, 0) for p in pts]
@show vars
# We're trying to fit the data points (x, y) = {(c1, 0), (c2, 0), (c3, 0)}
# using the function g_a(x) = (x-a)(x-2a)(x-4a)
R, v =  polynomial_ring(K, vars)

# # Initialise g_a(x) using a wrapper for absolute polynomials and their sums
g = PolydiscFunction([prod([(v[1]-v[i+1]) for i in 1:poly_degree])])
# Specify that x is the data variable and a is the parameter variable
f = AbstractModel(g, [(i == 1) for i in 1:(poly_degree+1)])
# ℓ is the loss function of this model wrt the L¹ norm 
ℓ = MPE_loss_init(1)

Initialising data
vars = ["x", "a_1", "a_2", "a_3", "a_4", "a_5", "a_6", "a_7"]


Loss(var"#MPE_compute#125"{Int64}(1), var"#MPE_grad#127"{Int64}(1))

In [3]:
println("Initialising optimisation tools")
# Let's pick the Gauss point as our initial parameter
gauss_point = ValuationPolydisc([K(1) for i in 1:poly_degree], zeros(Int, poly_degree))
model = Model(f, gauss_point)
# We optimise using Greedy descent
greedy_optim = greedy_descent_init(data, model, ℓ, 1)

Initialising optimisation tools


OptimSetup{PadicFieldElem, Int64, Int64}(Tuple{ValuationPolydisc{PadicFieldElem, Int64}, Int64}[(ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^2 + 2^3 + O(2^22)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^0 + 2^2 + 2^3 + 2^4 + O(2^20)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^2 + 2^3 + 2^4 + 2^5 + O(2^22)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^2 + 2^3 + 2^4 + 2^5 + O(2^22)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^1 + 2^2 + 2^3 + 2^4 + 2^5 + O(2^21)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^1 + 2^4 + O(2^21)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^0 + 2^1 + 2^2 + 2^5 + O(2^20)], [20]), 0)], Model{PadicFieldElem, Int64}(AbstractModel{PadicFieldElem}(PolydiscFunction{PadicFieldElem}(AbstractAlgebra.Generic.MPoly{PadicFieldElem}[x^7 + (2^0 + 2^1 + 2^2 + 2^3 + 2^4 + 2^5 + 2^6 + 2^7 + 2^8 + 2^9 + 2^

In [4]:
println("Optimising parameter")
# Compute the loss 

println("Loss before starting greedy descent is ", eval_loss(greedy_optim))

# Make 20 steps using greedy descent
N_epochs = 40
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim))
    step!(greedy_optim)
end 
t2 = time()

# Compute the new value of the loss
println("Greedy descent finished in ", t2-t1, " seconds. \nThe Final parameter a is ", greedy_optim.model.param)

# Now we compute the distances from parameters to roots of the polynomials. Notice that these might not come in the same orders
# so we need to take that into account when computing the distances. 

# First find the closest param to each root (for simplicity we allow repetitions in the params)
dist = [minimum([padic_abs(greedy_optim.model.param.center[i] - c[j]) for i in Base.eachindex(c)]) for j in Base.eachindex(c)]

# Then take max over that 
println("Distance of param to roots is at most ", maximum(dist))

Optimising parameter
Loss before starting greedy descent is 1.0
Loss at epoch 1 is 1.0
Loss at epoch 2 is 0.6428571428571428
Loss at epoch 3 is 0.46428571428571425
Loss at epoch 4 is 0.3214285714285714
Loss at epoch 5 is 0.23214285714285712
Loss at epoch 6 is 0.1607142857142857
Loss at epoch 7 is 0.11607142857142856
Loss at epoch 8 is 0.08035714285714285
Loss at epoch 9 is 0.06696428571428571
Loss at epoch 10 is 0.05803571428571428
Loss at epoch 11 is 0.049107142857142856
Loss at epoch 12 is 0.040178571428571425
Loss at epoch 13 is 0.033482142857142856
Loss at epoch 14 is 0.02901785714285714
Loss at epoch 15 is 0.024553571428571428
Loss at epoch 16 is 0.020089285714285712
Loss at epoch 17 is 0.016741071428571428
Loss at epoch 18 is 0.01450892857142857
Loss at epoch 19 is 0.012276785714285714
Loss at epoch 20 is 0.010044642857142856
Loss at epoch 21 is 0.008928571428571428
Loss at epoch 22 is 0.0078125
Loss at epoch 23 is 0.006696428571428571
Loss at epoch 24 is 0.006138392857142857
Los

In [5]:
# # Declare new model with same starting point as the first one
# new_model = Model(f, gauss_point)
# # Now let's optimise using gradient descent 
# gradient_optim = gradient_descent_init(data, new_model, ℓ, 3)

# println("Loss before starting gradient descent is ", eval_loss(gradient_optim))

# t3 = time()
# for i in 1:N_epochs
#     println("Loss at epoch ", i, " is ", eval_loss(gradient_optim))
#     step!(gradient_optim)
# end 
# t4 = time()

# # Compute the new value of the loss
# println("Gradient descent finished in ", t4-t3, "Seconds. \nThe Final parameter a is ", gradient_optim.model.param)